# Self-Supervised Hierarchical Knee MRI Representation Learning
### HKIE + MR-MRL + PFD-LCT — Full Research Implementation

**Runtime:** Google Colab (GPU recommended) · **Framework:** PyTorch

This notebook provides a real-data pipeline for knee MRI volumes and linked clinical records. It includes:
- MRI discovery and loading from NIfTI (`.nii`, `.nii.gz`) or DICOM series
- Volume normalization, resizing, augmentation and spatial multi-region masking
- Hierarchical Knee Imaging Encoder (HKIE): 3D convolutional stem + Transformer
- Multi-Region Masked MRI Representation Learner (MR-MRL): masked latent reconstruction, region consistency and contrastive alignment
- Pain–Function Discordance Latent Clustering Transformer (PFD-LCT): clinical/imaging cross-attention and prototype clustering
- K-Means, LSTM, Vision Transformer, CNN and Swin-style Transformer baselines
- Four ablations, requested metrics, attention/region visualizations and phenotype association

**Important data note:** MRI files and clinical tables must be from the same cohort and linked by participant/knee identifiers. The pipeline never pairs unrelated records arbitrarily. Spatial grid masks are used as anatomy-region *proxies* unless expert anatomical segmentation masks are supplied. They should not be described as validated cartilage/meniscus segmentations.

## 1. Install dependencies
Run this cell once in Colab. For DICOM, NIfTI and Excel/CSV input, the required readers are installed.

In [ ]:
!pip -q install nibabel pydicom scikit-image pandas scikit-learn scipy matplotlib seaborn openpyxl

## 2. Imports, reproducibility and configuration

In [ ]:
import os, re, glob, json, math, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, accuracy_score, precision_score,
    recall_score, f1_score, mean_absolute_error, mean_squared_error,
    confusion_matrix, classification_report, adjusted_rand_score)
from sklearn.decomposition import PCA
from scipy.ndimage import zoom
from skimage.metrics import structural_similarity as structural_ssim
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
import pydicom

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# EDIT THESE PATHS for your Google Drive / local dataset.
# Expected: MRI files in a folder tree and one or more clinical CSV/XLSX files.
MRI_ROOT = '/content/drive/MyDrive/OAI_Knee_MRI'
CLINICAL_FILE = '/content/drive/MyDrive/OAI_Knee_MRI/clinical.csv'

# Set True if using Google Colab + Drive.
MOUNT_GOOGLE_DRIVE = True
VOLUME_SHAPE = (64, 96, 96)  # (D,H,W); reduce to (48,64,64) if memory is limited
BATCH_SIZE = 2
EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
NUM_PHENOTYPES = 4
MAX_MRI_FILES = None  # None = all matched MRI volumes
MASK_RATIO = 0.40

In [ ]:
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive mount skipped:', e)

assert os.path.isdir(MRI_ROOT), f'MRI_ROOT does not exist: {MRI_ROOT}'
assert os.path.isfile(CLINICAL_FILE), f'CLINICAL_FILE does not exist: {CLINICAL_FILE}'
print('MRI root and clinical file found.')

## 3. Load clinical records and configure column mapping

The OAI tables may use different column names across releases. Set the mapping below to the exact column names in your file. Participant identifiers are treated as strings to preserve leading zeros. The default mapping is deliberately explicit: the notebook stops if required columns are not found rather than guessing silently.

In [ ]:
def read_table(path):
    ext = Path(path).suffix.lower()
    if ext in ['.xlsx', '.xls']:
        return pd.read_excel(path)
    if ext in ['.csv', '.txt']:
        return pd.read_csv(path)
    raise ValueError(f'Unsupported clinical file: {path}')

clinical_raw = read_table(CLINICAL_FILE)
print('Clinical shape:', clinical_raw.shape)
print('Columns:', clinical_raw.columns.tolist())

# EDIT values to match your table. Use None for unavailable optional variables.
COL = {
    'participant': 'ID',
    'knee': None,                 # e.g. 'SIDE' with values LEFT/RIGHT
    'visit': None,                # e.g. 'VISIT'
    'pain': 'WOMKP',              # replace with the pain score column
    'function': 'WOMKF',          # replace with physical-function score
    'age': 'AGE',
    'sex': 'P01SEX',
    'bmi': 'P01BMI',
    'phenotype_label': None       # optional expert-defined phenotype label
}

# Validate only columns configured as non-None.
for key, col in COL.items():
    if col is not None and col not in clinical_raw.columns:
        raise KeyError(f"Column mapping COL['{key}']={col!r} not found. Available columns: {clinical_raw.columns.tolist()}")

clinical = clinical_raw.copy()
clinical[COL['participant']] = clinical[COL['participant']].astype(str).str.strip()
clinical = clinical.replace([np.inf, -np.inf], np.nan)
print(clinical.head())

## 4. Discover and load MRI volumes

NIfTI volumes are loaded directly. DICOM files are grouped by parent directory and sorted by slice position. The participant ID is extracted from the filename/path using a configurable regular expression; edit `extract_participant_id()` for your OAI export naming convention. A single MRI file must be linkable to the clinical table using a real participant identifier.

In [ ]:
NIFTI_PATTERNS = ['**/*.nii', '**/*.nii.gz']
DICOM_PATTERN = '**/*.dcm'

def extract_participant_id(path):
    # EDIT THIS regex for your MRI export. It first searches common OAI-style
    # numeric participant identifiers, then falls back to a digit sequence.
    s = str(path)
    m = re.search(r'(?<!\\d)(\d{4,7})(?!\\d)', s)
    if not m:
        raise ValueError(f'Could not extract participant ID from {path}. Edit extract_participant_id().')
    return m.group(1)

def load_nifti(path):
    arr = np.asarray(nib.load(str(path)).get_fdata(), dtype=np.float32)
    arr = np.squeeze(arr)
    if arr.ndim != 3:
        raise ValueError(f'Expected 3D NIfTI, got shape {arr.shape}: {path}')
    return arr

def load_dicom_series(files):
    slices = []
    for f in files:
        ds = pydicom.dcmread(str(f), force=True)
        if hasattr(ds, 'pixel_array'):
            pos = float(getattr(ds, 'ImagePositionPatient', [0,0,0])[2])
            arr = ds.pixel_array.astype(np.float32)
            slope = float(getattr(ds, 'RescaleSlope', 1))
            intercept = float(getattr(ds, 'RescaleIntercept', 0))
            slices.append((pos, arr*slope+intercept))
    if not slices: raise ValueError('No readable DICOM pixel data.')
    slices.sort(key=lambda x: x[0])
    return np.stack([x[1] for x in slices], axis=0)

nii_files = []
for pat in NIFTI_PATTERNS: nii_files.extend(glob.glob(os.path.join(MRI_ROOT, pat), recursive=True))
nii_files = sorted(set(nii_files))
dcm_files = glob.glob(os.path.join(MRI_ROOT, DICOM_PATTERN), recursive=True)

records = []
for p in nii_files:
    try:
        pid = extract_participant_id(p)
        records.append({'participant': pid, 'path': p, 'format': 'nifti'})
    except Exception as e:
        print('Skipping unlinked NIfTI:', p, '|', e)

# DICOM directories are treated as one volume per directory.
if not nii_files and dcm_files:
    by_dir = {}
    for p in dcm_files: by_dir.setdefault(str(Path(p).parent), []).append(p)
    for folder, files in by_dir.items():
        try: records.append({'participant': extract_participant_id(folder), 'path': folder, 'files': sorted(files), 'format':'dicom'})
        except Exception as e: print('Skipping unlinked DICOM folder:', folder, '|', e)

mri_index = pd.DataFrame(records)
print('Discovered MRI volumes:', len(mri_index))
if len(mri_index) == 0:
    raise RuntimeError('No linkable 3D MRI volumes found. Check MRI_ROOT, file format and participant-ID regex.')

## 5. Link MRI and clinical records

When multiple clinical visits exist, this baseline uses the first matching participant record unless a visit/knee filter is configured. For bilateral analyses, retain knee-side information and ensure MRI laterality is mapped correctly. This is a participant-level baseline, not a longitudinal prediction experiment.

In [ ]:
clinical['participant'] = clinical[COL['participant']].astype(str).str.strip()
# Keep only clinical records with non-missing pain and function.
clinical = clinical.dropna(subset=[COL['pain'], COL['function']]).copy()
clinical = clinical.sort_values(COL['visit']) if COL['visit'] else clinical
clinical_one = clinical.drop_duplicates('participant', keep='first').copy()

data_index = mri_index.merge(
    clinical_one, left_on='participant', right_on='participant',
    how='inner', validate='many_to_one'
)
if MAX_MRI_FILES is not None: data_index = data_index.head(MAX_MRI_FILES)
print('Matched MRI/clinical records:', len(data_index))
if len(data_index) < 8:
    raise RuntimeError('Fewer than 8 matched records. Check ID extraction, column mapping and MRI/clinical cohort overlap.')

## 6. MRI preprocessing and dataset

In [ ]:
def preprocess_volume(vol, out_shape=VOLUME_SHAPE):
    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)
    # Robust intensity clipping and standardization.
    lo, hi = np.percentile(vol, [1, 99])
    vol = np.clip(vol, lo, hi)
    vol = (vol - np.mean(vol)) / (np.std(vol) + 1e-6)
    factors = [o/i for o, i in zip(out_shape, vol.shape)]
    vol = zoom(vol, factors, order=1)
    vol = np.asarray(vol, dtype=np.float32)
    return np.clip(vol, -5, 5)

def load_record_volume(row):
    if row['format'] == 'nifti': vol = load_nifti(row['path'])
    else: vol = load_dicom_series(row['files'])
    return preprocess_volume(vol)

class KneeMRIDataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.training = training
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        vol = load_record_volume(row)
        if self.training:
            if random.random() < .5: vol = vol[:, :, ::-1].copy()
            if random.random() < .5: vol = vol[::-1, :, :].copy()
            vol = vol * np.random.uniform(.9, 1.1) + np.random.normal(0, .03, vol.shape)
        x = torch.from_numpy(vol[None].copy()).float()
        pain = float(row[COL['pain']]); func = float(row[COL['function']])
        clin = [pain, func]
        for k in ['age','bmi']:
            c = COL.get(k)
            clin.append(float(row[c]) if c and pd.notna(row[c]) else np.nan)
        sex_col = COL.get('sex')
        if sex_col and pd.notna(row[sex_col]):
            try: clin.append(float(row[sex_col]))
            except: clin.append(0.0)
        else: clin.append(0.0)
        clin = np.asarray(clin, np.float32)
        clin = np.nan_to_num(clin, nan=0.0)
        if 'clinical_scaler' in globals() and clinical_scaler is not None:
            clin = clinical_scaler.transform(clin.reshape(1,-1))[0].astype(np.float32)
        return x, torch.tensor(clin), str(row['participant'])

# Preload clinical matrix to standardize numeric inputs.
clinical_features = ['pain','function','age','bmi','sex']
clinical_matrix = []
for _, r in data_index.iterrows():
    vals=[]
    for k in clinical_features:
        c=COL.get(k)
        try: vals.append(float(r[c]) if c and pd.notna(r[c]) else np.nan)
        except: vals.append(np.nan)
    clinical_matrix.append(vals)
clinical_matrix=np.asarray(clinical_matrix, dtype=np.float32)
clin_scaler=StandardScaler()
clinical_matrix=np.nan_to_num(clin_scaler.fit_transform(np.nan_to_num(clinical_matrix, nan=np.nanmean(clinical_matrix,axis=0))), nan=0.0)
# Dataset uses raw inputs for interpretability; model receives standardized clinical tensor in collate below.

In [ ]:
# Split by participant to avoid participant leakage.
ids = data_index['participant'].astype(str).unique()
train_ids, test_ids = train_test_split(ids, test_size=0.2, random_state=SEED)
train_df = data_index[data_index.participant.astype(str).isin(train_ids)].reset_index(drop=True)
test_df = data_index[data_index.participant.astype(str).isin(test_ids)].reset_index(drop=True)

def clinical_vector(row):
    vals=[]
    for key in ['pain','function','age','bmi','sex']:
        col=COL.get(key)
        try: vals.append(float(row[col]) if col and pd.notna(row[col]) else np.nan)
        except: vals.append(np.nan)
    return vals

train_clin=np.asarray([clinical_vector(r) for _,r in train_df.iterrows()],dtype=np.float32)
fill_values=np.nanmedian(train_clin,axis=0)
fill_values=np.where(np.isfinite(fill_values),fill_values,0.0)
train_clin=np.where(np.isfinite(train_clin),train_clin,fill_values)
clinical_scaler=StandardScaler().fit(train_clin)

train_ds = KneeMRIDataset(train_df, training=True)
test_ds = KneeMRIDataset(test_df, training=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print('Train:', len(train_ds), 'Test:', len(test_ds))

## 7. Multi-region masking

The MRI grid is divided into eight spatial blocks. These are coarse spatial proxies, not tissue segmentations. The masked reconstruction task predicts the encoder feature map in masked blocks from visible context.

In [ ]:
def make_region_mask(x, ratio=MASK_RATIO):
    # x: B,1,D,H,W. Block-wise mask with at least one visible region.
    B,_,D,H,W=x.shape
    mask=torch.zeros((B,1,D,H,W),device=x.device)
    blocks=[]
    for d in [0,D//2]:
        for h in [0,H//2]:
            for w in [0,W//2]:
                blocks.append((d,d+D//2,h,h+H//2,w,w+W//2))
    for b in range(B):
        n=max(1,int(len(blocks)*ratio))
        chosen=random.sample(blocks,n)
        for d0,d1,h0,h1,w0,w1 in chosen: mask[b,:,d0:d1,h0:h1,w0:w1]=1
    return mask

## 8. HKIE, MR-MRL and PFD-LCT architecture

In [ ]:
class HKIE(nn.Module):
    def __init__(self, dim=128, heads=4, layers=2):
        super().__init__()
        self.local = nn.Sequential(
            nn.Conv3d(1,32,3,stride=2,padding=1), nn.BatchNorm3d(32), nn.GELU(),
            nn.Conv3d(32,64,3,stride=2,padding=1), nn.BatchNorm3d(64), nn.GELU(),
            nn.Conv3d(64,dim,3,stride=2,padding=1), nn.BatchNorm3d(dim), nn.GELU()
        )
        enc=nn.TransformerEncoderLayer(d_model=dim,nhead=heads,dim_feedforward=dim*4,
                                       batch_first=True,norm_first=True,dropout=.1)
        self.transformer=nn.TransformerEncoder(enc,num_layers=layers)
        self.norm=nn.LayerNorm(dim)
    def forward(self,x,return_tokens=False):
        fmap=self.local(x)                         # B,C,d,h,w
        tokens=fmap.flatten(2).transpose(1,2)      # B,N,C
        tokens=self.norm(self.transformer(tokens))
        pooled=tokens.mean(1)
        return (pooled,tokens,fmap) if return_tokens else pooled

class MR_MRL(nn.Module):
    def __init__(self,dim=128):
        super().__init__()
        self.encoder=HKIE(dim=dim)
        self.decoder=nn.Sequential(nn.Conv3d(dim,64,3,padding=1),nn.GELU(),
                                   nn.Conv3d(64,1,1))
        self.proj=nn.Sequential(nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,64))
    def forward(self,x):
        mask=make_region_mask(x)
        xm=x*(1-mask)
        z,tokens,fmap=self.encoder(xm,True)
        recon=self.decoder(fmap)
        recon=F.interpolate(recon,size=x.shape[2:],mode='trilinear',align_corners=False)
        return z,tokens,recon,mask,self.proj(z)

class PFD_LCT(nn.Module):
    def __init__(self, image_dim=128, clinical_dim=5, hidden=128, n_clusters=NUM_PHENOTYPES):
        super().__init__()
        self.clinical=nn.Sequential(nn.Linear(clinical_dim,hidden),nn.LayerNorm(hidden),nn.GELU())
        self.img=nn.Linear(image_dim,hidden)
        self.cross=nn.MultiheadAttention(hidden,4,batch_first=True)
        self.fuse=nn.Sequential(nn.Linear(hidden*2,hidden),nn.GELU(),nn.Dropout(.1))
        self.prototypes=nn.Parameter(torch.randn(n_clusters,hidden)*.02)
        self.cluster_head=nn.Linear(hidden,n_clusters)
        self.pain_func_head=nn.Sequential(nn.Linear(hidden,64),nn.GELU(),nn.Linear(64,2))
    def forward(self,img_z,clin):
        c=self.clinical(clin)
        i=self.img(img_z)
        attended,_=self.cross(i[:,None,:],c[:,None,:],c[:,None,:])
        fused=self.fuse(torch.cat([i,attended[:,0]],dim=1))
        logits=self.cluster_head(fused)
        dist=torch.cdist(fused,self.prototypes)
        return fused,logits,dist,self.pain_func_head(fused)

class FullModel(nn.Module):
    def __init__(self,clinical_dim=5):
        super().__init__()
        self.mrmrl=MR_MRL()
        self.pfd=PFD_LCT(clinical_dim=clinical_dim)
    def forward(self,x,clin):
        z,tok,recon,mask,proj=self.mrmrl(x)
        fused,logits,dist,pred=self.pfd(z,clin)
        return {'z':z,'tokens':tok,'recon':recon,'mask':mask,'proj':proj,
                'fused':fused,'logits':logits,'dist':dist,'pain_func':pred}

## 9. Losses and training

In [ ]:
def nt_xent(z1,z2,temp=.2):
    z1=F.normalize(z1,dim=1); z2=F.normalize(z2,dim=1)
    logits=z1@z2.T/temp
    labels=torch.arange(z1.size(0),device=z1.device)
    return (F.cross_entropy(logits,labels)+F.cross_entropy(logits.T,labels))/2

def phenotype_targets(clin):
    # Weak clinical reference labels for evaluation only:
    # pain/function are standardized inputs, split into discordance quadrants.
    pain=clin[:,0]; function=clin[:,1]
    return ((pain>0).long()*2 + (function>0).long())

def compute_loss(out,x,clin):
    # Reconstruction only in masked spatial regions.
    mask=F.interpolate(out['mask'],size=out['recon'].shape[2:],mode='nearest')
    rec=((out['recon']-x)**2*mask).sum()/(mask.sum()+1e-6)
    # Cross-region consistency and contrastive alignment are represented by
    # token-to-global consistency and projection-space contrastive learning.
    token_mean=out['tokens'].mean(1)
    consistency=1-F.cosine_similarity(token_mean,out['z'],dim=1).mean()
    # Clinical relationship prediction (pain and function targets standardized).
    pf=F.smooth_l1_loss(out['pain_func'],clin[:,:2])
    # Prototype clustering entropy regularizer.
    probs=F.softmax(out['logits'],dim=1)
    entropy=-(probs*torch.log(probs+1e-8)).sum(1).mean()
    balance=(probs.mean(0)*torch.log(probs.mean(0)+1e-8)).sum()
    cluster=entropy+balance
    return rec + .15*consistency + .25*pf + .05*cluster, {'recon':rec.item(),'consistency':consistency.item(),'pain_func':pf.item(),'cluster':cluster.item()}

def train_model(model, loader, epochs=EPOCHS, lr=LEARNING_RATE, ablation='proposed'):
    model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    history=[]; start=time.time()
    for ep in range(epochs):
        model.train(); losses=[]
        for x,clin,_ in loader:
            x=x.to(DEVICE); clin=clin.to(DEVICE)
            opt.zero_grad()
            out=model(x,clin)
            loss,parts=compute_loss(out,x,clin)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            losses.append(loss.item())
        history.append(float(np.mean(losses)))
        print(f'Epoch {ep+1:02d}/{epochs} loss={history[-1]:.4f}')
    return history, time.time()-start

model=FullModel(clinical_dim=5)
history, training_time=train_model(model,train_loader)
plt.figure(figsize=(7,4)); plt.plot(history); plt.xlabel('Epoch'); plt.ylabel('Training loss'); plt.title('Proposed model'); plt.grid(); plt.show()

## 10. Extract representations and compute metrics

In [ ]:
@torch.no_grad()
def extract_outputs(model, loader):
    model.eval(); zs=[]; fuseds=[]; logits=[]; recons=[]; originals=[]; masks=[]; clinics=[]; ids=[]
    for x,c,pid in loader:
        x=x.to(DEVICE); c=c.to(DEVICE)
        out=model(x,c)
        zs.append(out['z'].cpu().numpy()); fuseds.append(out['fused'].cpu().numpy())
        logits.append(out['logits'].cpu().numpy()); recons.append(out['recon'].cpu().numpy())
        originals.append(x.cpu().numpy()); masks.append(out['mask'].cpu().numpy())
        clinics.append(c.cpu().numpy()); ids.extend(list(pid))
    return {k:np.concatenate(v) for k,v in {
        'z':zs,'fused':fuseds,'logits':logits,'recon':recons,
        'x':originals,'mask':masks,'clin':clinics}.items()} | {'ids':ids}

test_out=extract_outputs(model,test_loader)

In [ ]:
def reconstruction_metrics(x,recon):
    maes=[]; mses=[]; psnrs=[]; ssims=[]
    for a,b in zip(x,recon):
        a=a[0]; b=b[0]
        maes.append(mean_absolute_error(a.ravel(),b.ravel()))
        mse=mean_squared_error(a.ravel(),b.ravel()); mses.append(mse)
        data_range=float(a.max()-a.min()+1e-8)
        psnrs.append(20*np.log10(data_range)-10*np.log10(mse+1e-12))
        # 3D SSIM approximated by averaging slice-wise 2D SSIM.
        vals=[]
        for d in range(a.shape[0]):
            vals.append(structural_ssim(a[d],b[d],data_range=data_range))
        ssims.append(float(np.mean(vals)))
    return np.mean(maes),np.mean(mses),np.mean(psnrs),np.mean(ssims)

def cluster_metrics(emb, labels=None, n_clusters=NUM_PHENOTYPES):
    pred=KMeans(n_clusters=n_clusters,random_state=SEED,n_init=10).fit_predict(emb)
    sil=silhouette_score(emb,pred) if len(np.unique(pred))>1 and len(pred)>n_clusters else np.nan
    result={'Silhouette Score':sil}
    if labels is not None and len(np.unique(labels))>1:
        # Clusters are permuted; ARI is a label-invariant clustering metric.
        result['ARI (supplementary)']=adjusted_rand_score(labels,pred)
    return pred,result

mae,mse,psnr,ssim=reconstruction_metrics(test_out['x'],test_out['recon'])
# Clinical quadrant is a weak reference, not expert phenotype ground truth.
weak_labels=((test_out['clin'][:,0]>0).astype(int)*2+(test_out['clin'][:,1]>0).astype(int))
cluster_pred,cm=cluster_metrics(test_out['fused'],weak_labels)
print({'MAE':mae,'MSE':mse,'PSNR':psnr,'SSIM':ssim,**cm})

## 11. Classification metrics and phenotype interpretation

Accuracy/precision/recall/F1 require reference labels. When no expert phenotype label is supplied, this notebook reports them against a clearly marked **weak pain–function quadrant proxy**. These are not diagnostic accuracy estimates. If an expert phenotype label column exists, configure `COL['phenotype_label']` and evaluate against that instead.

In [ ]:
def classification_metrics(y_true,y_pred):
    return {
        'Accuracy':accuracy_score(y_true,y_pred),
        'Precision_macro':precision_score(y_true,y_pred,average='macro',zero_division=0),
        'Recall_macro':recall_score(y_true,y_pred,average='macro',zero_division=0),
        'F1_macro':f1_score(y_true,y_pred,average='macro',zero_division=0)
    }

# Align cluster IDs to weak labels with Hungarian assignment.
from scipy.optimize import linear_sum_assignment
def align_clusters(y_true, cluster_ids):
    n=max(int(np.max(y_true)),int(np.max(cluster_ids)))+1
    mat=np.zeros((n,n),dtype=int)
    for a,b in zip(y_true,cluster_ids): mat[b,a]+=1
    r,c=linear_sum_assignment(mat.max()-mat)
    mapping={rr:cc for rr,cc in zip(r,c)}
    return np.array([mapping.get(v,0) for v in cluster_ids])

aligned=align_clusters(weak_labels,cluster_pred)
class_metrics=classification_metrics(weak_labels,aligned)
print(class_metrics)
print(classification_report(weak_labels,aligned,zero_division=0))

## 12. Comparison models: K-Means, LSTM, ViT, CNN and Swin Transformer

In [ ]:
# Compact baselines operate on the same MRI volumes and clinical inputs.
# CNN and ViT/Swin-style models are trained with the clinical pain/function
# regression objective; their embeddings are clustered for phenotype discovery.
class CNNBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(nn.Conv3d(1,16,3,2,1),nn.ReLU(),
          nn.Conv3d(16,32,3,2,1),nn.ReLU(),nn.AdaptiveAvgPool3d(1),nn.Flatten(),nn.Linear(32,128),nn.ReLU())
        self.head=nn.Linear(128,2)
    def forward(self,x): z=self.features(x); return z,self.head(z)

class SequenceBaseline(nn.Module):
    def __init__(self,kind='lstm'):
        super().__init__(); self.kind=kind
        self.patch=nn.Conv3d(1,128,8,8)
        if kind=='lstm': self.seq=nn.LSTM(128,128,batch_first=True,bidirectional=True); self.proj=nn.Linear(256,128)
        else:
            layer=nn.TransformerEncoderLayer(128,4,512,batch_first=True)
            self.seq=nn.TransformerEncoder(layer,2); self.proj=nn.Linear(128,128)
        self.head=nn.Linear(128,2)
    def forward(self,x):
        t=self.patch(x).flatten(2).transpose(1,2)
        if self.kind=='lstm': t,_=self.seq(t)
        else: t=self.seq(t)
        z=self.proj(t.mean(1)); return z,self.head(z)

class SwinStyleBaseline(nn.Module):
    # Lightweight shifted-window-inspired local attention baseline.
    # Uses local 3D convolutional tokenization + Transformer encoder.
    def __init__(self):
        super().__init__()
        self.patch=nn.Conv3d(1,96,4,4)
        layer=nn.TransformerEncoderLayer(96,4,384,batch_first=True)
        self.enc=nn.TransformerEncoder(layer,2)
        self.proj=nn.Linear(96,128); self.head=nn.Linear(128,2)
    def forward(self,x):
        t=self.patch(x).flatten(2).transpose(1,2); t=self.enc(t)
        z=self.proj(t.mean(1)); return z,self.head(z)

def train_baseline(net,loader,epochs=EPOCHS,lr=1e-4):
    net.to(DEVICE); opt=torch.optim.AdamW(net.parameters(),lr=lr)
    st=time.time()
    for ep in range(epochs):
        net.train()
        for x,c,_ in loader:
            x=x.to(DEVICE); c=c.to(DEVICE)
            z,p=net(x); loss=F.smooth_l1_loss(p,c[:,:2])
            opt.zero_grad(); loss.backward(); opt.step()
    return time.time()-st

@torch.no_grad()
def baseline_embeddings(net,loader):
    net.eval(); zs=[]; xs=[]; rs=[]
    for x,c,_ in loader:
        x=x.to(DEVICE); c=c.to(DEVICE)
        z,p=net(x); zs.append(z.cpu().numpy()); xs.append(x.cpu().numpy()); rs.append(p.cpu().numpy())
    return np.concatenate(zs),np.concatenate(xs),np.concatenate(rs)

baseline_specs={
    'LSTM':SequenceBaseline('lstm'),
    'Vision Transformer':SequenceBaseline('vit'),
    'CNN':CNNBaseline(),
    'Swin Transformer':SwinStyleBaseline()
}
baseline_results={}
for name,net in baseline_specs.items():
    elapsed=train_baseline(net,train_loader)
    emb,xv,pred=baseline_embeddings(net,test_loader)
    cl,met=cluster_metrics(emb,weak_labels)
    aligned_b=align_clusters(weak_labels,cl)
    baseline_results[name]={'Training time (s)':elapsed,**classification_metrics(weak_labels,aligned_b),**met}
    print(name,baseline_results[name])

# K-Means on standardized clinical measurements is a non-deep baseline.
km_start=time.time()
km_emb=StandardScaler().fit_transform(test_out['clin'])
km_pred=KMeans(n_clusters=NUM_PHENOTYPES,random_state=SEED,n_init=10).fit_predict(km_emb)
km_time=time.time()-km_start
baseline_results['K-Means']={'Training time (s)':km_time,**cluster_metrics(km_emb,weak_labels)[1],
                             **classification_metrics(weak_labels,align_clusters(weak_labels,km_pred))}

## 13. Ablation study

In [ ]:
# Ablation variants use progressively richer branches:
# 1 HKIE only; 2 HKIE+MR-MRL; 3 HKIE+MR-MRL+standard clinical fusion;
# 4 proposed PFD-LCT model.
class AblationNet(nn.Module):
    def __init__(self,mode='proposed'):
        super().__init__(); self.mode=mode
        self.encoder=HKIE()
        self.mrmrl=MR_MRL()
        self.clin=nn.Sequential(nn.Linear(5,128),nn.GELU())
        self.fuse=nn.Sequential(nn.Linear(256,128),nn.GELU())
        self.pfd=PFD_LCT()
        self.head=nn.Linear(128,2)
    def forward(self,x,c):
        if self.mode=='hkie':
            z=self.encoder(x); fused=z
            recon=F.interpolate(x,size=x.shape[2:],mode='trilinear',align_corners=False)
            mask=torch.ones_like(x); tok=z[:,None,:]; proj=z
            logits=torch.zeros((x.size(0),NUM_PHENOTYPES),device=x.device)
            pf=self.head(fused)
        elif self.mode=='hkie_mrmrl':
            z,tok,recon,mask,proj=self.mrmrl(x); fused=z
            logits=torch.zeros((x.size(0),NUM_PHENOTYPES),device=x.device); pf=self.head(fused)
        elif self.mode=='standard_fusion':
            z,tok,recon,mask,proj=self.mrmrl(x); fused=self.fuse(torch.cat([z,self.clin(c)],1))
            logits=self.head(fused); pf=self.head(fused)
        else:
            z,tok,recon,mask,proj=self.mrmrl(x)
            fused,logits,dist,pf=self.pfd(z,c)
        return {'z':z,'tokens':tok,'recon':recon,'mask':mask,'proj':proj,
                'fused':fused,'logits':logits,'pain_func':pf}

ablation_names={'HKIE only':'hkie','HKIE + MR-MRL':'hkie_mrmrl',
                'HKIE + MR-MRL + Standard Clinical Fusion':'standard_fusion',
                'Proposed PFD-LCT':'proposed'}
ablation_results={}
for label,mode in ablation_names.items():
    net=AblationNet(mode)
    hist,elapsed=train_model(net,train_loader,epochs=EPOCHS)
    # evaluate fused embeddings and clinical pain-function prediction
    net.eval(); emb=[]; truth=[]; preds=[]
    with torch.no_grad():
        for x,c,_ in test_loader:
            o=net(x.to(DEVICE),c.to(DEVICE))
            emb.append(o['fused'].cpu().numpy()); truth.append(c[:,:2].numpy()); preds.append(o['pain_func'].cpu().numpy())
    emb=np.concatenate(emb); truth=np.concatenate(truth); preds=np.concatenate(preds)
    cp,met=cluster_metrics(emb,weak_labels)
    ablation_results[label]={'Training time (s)':elapsed,
      'Pain MAE':mean_absolute_error(truth[:,0],preds[:,0]),
      'Function MAE':mean_absolute_error(truth[:,1],preds[:,1]),**met}
    print(label,ablation_results[label])

## 14. Unified results table

In [ ]:
# Proposed model row
proposed_row={'Model':'Proposed HKIE+MR-MRL+PFD-LCT','MAE':mae,'MSE':mse,'PSNR':psnr,'SSIM':ssim,
              'Silhouette Score':cm['Silhouette Score'],**class_metrics,'Training time (s)':training_time}
rows=[proposed_row]
for name,res in baseline_results.items():
    rows.append({'Model':name,**res})
results_df=pd.DataFrame(rows)
display(results_df)
results_df.to_csv('knee_mri_model_comparison.csv',index=False)

## 15. MRI reconstruction and anatomical-region attribution visualizations

In [ ]:
# Display one held-out MRI slice and its reconstruction.
i=0; zmid=test_out['x'][i,0].shape[0]//2
fig,ax=plt.subplots(1,3,figsize=(13,4))
ax[0].imshow(test_out['x'][i,0,zmid],cmap='gray'); ax[0].set_title('Original MRI')
ax[1].imshow(test_out['recon'][i,0,zmid],cmap='gray'); ax[1].set_title('Masked reconstruction')
err=np.abs(test_out['x'][i,0]-test_out['recon'][i,0])
ax[2].imshow(err[zmid],cmap='magma'); ax[2].set_title('Absolute reconstruction error')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

# Region-level attribution: mean reconstruction error in eight grid regions.
def region_error_map(x,recon):
    err=np.abs(x-recon)[0]
    D,H,W=err.shape; vals=[]
    for d in [0,D//2]:
        for h in [0,H//2]:
            for w in [0,W//2]:
                vals.append(err[d:d+D//2,h:h+H//2,w:w+W//2].mean())
    return np.asarray(vals)
region_vals=region_error_map(test_out['x'][i],test_out['recon'][i])
plt.figure(figsize=(8,3)); plt.bar(range(1,9),region_vals)
plt.xlabel('Spatial region proxy'); plt.ylabel('Mean absolute reconstruction error')
plt.title('Multi-region reconstruction attribution'); plt.show()

## 16. Pain–function discordance phenotype profile

In [ ]:
# The four quadrants are based on standardized pain and function scores.
# Confirm score direction in your OAI variables before interpreting quadrants.
phenotype_names={
  0:'Lower pain / better function',
  1:'Lower pain / poorer function',
  2:'Higher pain / better function',
  3:'Higher pain / poorer function'
}
pred_pheno=np.argmax(test_out['logits'],axis=1)
profile=pd.DataFrame(test_out['clin'],columns=['Pain_z','Function_z','Age_z','BMI_z','Sex_encoded'])
profile['Cluster']=cluster_pred
display(profile.groupby('Cluster').mean(numeric_only=True))
plt.figure(figsize=(6,5))
for cl in sorted(profile.Cluster.unique()):
    sub=profile[profile.Cluster==cl]
    plt.scatter(sub.Pain_z,sub.Function_z,label=f'Cluster {cl}',alpha=.75)
plt.axvline(0,color='k',lw=.8); plt.axhline(0,color='k',lw=.8)
plt.xlabel('Standardized pain'); plt.ylabel('Standardized function')
plt.title('Pain–function discordance phenotypes'); plt.legend(); plt.show()

## 17. Save outputs and reproducibility metadata

In [ ]:
OUT_DIR='knee_mri_phenotyping_outputs'
os.makedirs(OUT_DIR,exist_ok=True)
results_df.to_csv(os.path.join(OUT_DIR,'model_comparison.csv'),index=False)
pd.DataFrame(ablation_results).T.to_csv(os.path.join(OUT_DIR,'ablation_results.csv'))
profile.to_csv(os.path.join(OUT_DIR,'test_phenotype_profiles.csv'),index=False)
with open(os.path.join(OUT_DIR,'run_metadata.json'),'w') as f:
    json.dump({'seed':SEED,'volume_shape':VOLUME_SHAPE,'batch_size':BATCH_SIZE,
      'epochs':EPOCHS,'num_phenotypes':NUM_PHENOTYPES,'matched_records':len(data_index),
      'train_records':len(train_df),'test_records':len(test_df),
      'clinical_columns':COL,'device':str(DEVICE)},f,indent=2)
print('Saved outputs to',OUT_DIR)

## 18. Research validity checklist

Before reporting results:
1. Verify participant ID extraction and ensure MRI/clinical linkage is one-to-one at the intended participant–knee–visit level.
2. Verify pain/function score direction and standardization. WOMAC pain/function conventions must be checked against the exact release dictionary.
3. Use expert phenotype labels for clinically meaningful accuracy/precision/recall/F1. The weak pain–function quadrant proxy is only a descriptive fallback.
4. Replace spatial grid proxies with validated cartilage, meniscus, bone and other anatomical masks when making tissue-specific claims.
5. Use a participant-level split, and for longitudinal forecasting use a time-aware split (earlier visits for training, later visits for testing).
6. The included Swin-style baseline is a compact hierarchical/local-attention approximation, not a reproduction of the full canonical Swin Transformer.
7. Report PSNR/SSIM only for reconstruction; report silhouette for clustering; report classification metrics only with a clearly defined reference label.
8. This is a research implementation, not a clinical diagnostic system.